# 05 Extended analysis
No new training: shortcut baselines, full explanation table (incl. deletion AUC), per-galaxy breakdowns, energy components. Code in `src/extended.py`.

In [1]:
import sys, warnings; sys.path.insert(0,'../src'); warnings.filterwarnings('ignore')
import pandas as pd, extended as E, data as D
meta=D.load_metadata(True); x=pd.read_csv(E.OUT/'xai_per_galaxy.csv')

In [2]:
E.shortcut_baselines().round(3)

,sample,features,LR,LR_sd,GBT,GBT_sd
0,unmatched,"Catalogue: $M_\star$, $f_{\rm gas}$, $z$",0.741,0.008,0.746,0.014
1,unmatched,"Catalogue + sSFR, $r_{\rm gas}/r_\star$",0.793,0.003,0.838,0.014
2,unmatched,"Image: gas area, total gas",0.739,0.014,0.853,0.019
3,unmatched,All seven scalars,0.829,0.014,0.870,0.030
4,matched,"Catalogue: $M_\star$, $f_{\rm gas}$, $z$",0.577,0.021,0.550,0.014
5,matched,"Catalogue + sSFR, $r_{\rm gas}/r_\star$",0.735,0.013,0.789,0.010
6,matched,"Image: gas area, total gas",0.590,0.018,0.804,0.007
7,matched,All seven scalars,0.755,0.023,0.813,0.006


In [3]:
t=E.full_xai_table(x); t.to_csv(E.OUT/'xai_full.csv',index=False); t.round(3)

,model,method,TEF,GEF,TEL,TEL_seed_sd,DEL,DEL_lo,DEL_hi
0,resnet50,gradcam,0.205,0.226,2.632,0.317,0.529,0.499,0.560
1,resnet50,gradcam++,0.224,0.249,2.954,0.249,0.499,0.470,0.530
2,resnet50,scorecam,0.262,0.292,3.573,0.116,0.409,0.383,0.437
3,resnet18,gradcam,0.204,0.226,2.566,0.025,0.609,0.586,0.633
4,resnet18,gradcam++,0.210,0.233,2.704,0.023,0.591,0.567,0.617
5,resnet18,scorecam,0.229,0.254,3.000,0.014,0.574,0.550,0.600
6,efficientnet_b0,gradcam,0.102,0.111,1.075,0.199,0.665,0.630,0.698
7,efficientnet_b0,gradcam++,0.162,0.169,1.639,0.411,0.565,0.530,0.599
8,efficientnet_b0,scorecam,0.168,0.185,2.382,0.225,0.529,0.493,0.566
9,mobilenetv3_small_100,gradcam,0.110,0.129,1.162,0.500,0.627,0.593,0.658


In [4]:
gc,area,sr,corr,nw,edges,ns=E.per_galaxy(x,meta)
E.fig_per_galaxy(area,sr,ns)
display(area.round(2), sr.round(3), corr.round(2), nw, edges)

area_bin,small,medium,large
model,,,
efficientnet_b0,0.93,1.01,1.28
mobilenetv3_small_100,0.86,1.31,1.32
resnet18,2.78,2.63,2.29
resnet50,2.99,2.58,2.32
small_cnn,7.54,4.94,2.61
vit_tiny_patch16_224,1.96,1.66,1.22


score_bin,0.80-0.90,0.90-0.97,>0.97
model,,,
efficientnet_b0,0.860,0.922,0.949
mobilenetv3_small_100,0.892,0.928,0.949
resnet18,0.946,0.939,0.949
resnet50,0.882,0.922,0.936
small_cnn,0.704,0.800,0.795
vit_tiny_patch16_224,0.919,0.944,0.962


correct,False,True
model,,
efficientnet_b0,0.96,1.09
mobilenetv3_small_100,0.81,1.19
resnet18,0.82,2.67
resnet50,0.64,2.84
small_cnn,2.73,5.78
vit_tiny_patch16_224,0.81,1.67


model
efficientnet_b0          14.666667
mobilenetv3_small_100    12.333333
resnet18                  8.333333
resnet50                 13.666667
small_cnn                35.666667
vit_tiny_patch16_224      9.333333
dtype: float64

,min,max
area_bin,,
small,0.002319,0.037476
medium,0.038879,0.094238
large,0.094971,0.523438


In [5]:
E.fig_data_check()

{'fg_med_un': (-0.526914915629652, 0.270100707461181),
 'fg_med_ma': (-0.527290050046009, -0.3800083627387826),
 'area_med_ma': (0.079437255859375, 0.040740966796875),
 'area_full_frac': (0.0020161290322580645, 0.14213709677419356)}

In [6]:
E.energy_breakdown().round(3)

,train_cpu_joules,train_gpu_joules,train_dram_joules,train_ane_joules,train_joules,train_mean_watts,epochs,train_seconds,j_per_epoch,cpu_frac,gpu_frac,dram_frac,ane_frac
model,,,,,,,,,,,,,
resnet50,127.758,1957.211,463.332,35.731,2584.031,23.253,17.000,111.137,152.002,0.049,0.757,0.179,0.014
resnet18,35.019,688.188,106.766,0.000,829.973,23.323,17.667,35.573,46.980,0.042,0.829,0.129,0.000
efficientnet_b0,87.348,1305.024,271.795,0.000,1664.166,18.907,17.667,88.025,94.198,0.052,0.784,0.163,0.000
mobilenetv3_small_100,57.594,291.073,45.864,0.000,394.531,17.457,19.333,22.374,20.407,0.146,0.738,0.116,0.000
vit_tiny_patch16_224,55.719,565.371,102.563,0.000,723.653,24.063,16.667,30.069,43.419,0.077,0.781,0.142,0.000
small_cnn,18.716,394.202,104.504,0.002,517.423,21.847,18.333,23.699,28.223,0.036,0.762,0.202,0.000
